# Codex

> ChatGPT subscription tokens for the Codex backend


In [ ]:
#| default_exp codex

## Imports

In [ ]:
import asyncio, base64, json, sys, time, httpx2
from fastcore.aio import wait_until
from fastcore.utils import *
from fastllm.types import vendor_auth, vendor_hdrs


In [ ]:
#| hide
from fastcore.test import *
from fastcore.funccall import get_schema
from cachy.core import enable_cachy
from aidialog.msg_parts import mk_msg, mk_tool_res_msg
from fastllm.types import luna
from fastllm.acomplete import acomplete, mk_client, defaults, vendor_doms


In [ ]:
#| hide
enable_cachy(doms=vendor_doms, hdrs=('content-type',))
defaults.reasoning_effort = 'l'

FastLLM's Codex backend uses a ChatGPT subscription token, not an API key. With file-based Codex login, `acomplete` reads it from `~/.codex/auth.json` through `codex_auth`.

This module reads auth data, inspects token claims, refreshes expired tokens, and starts device-code sign-ins. Most helpers accept plain dicts. A server storing tokens for many users can use them without a local auth file.

## Tokens

In [ ]:
_CLIENT_ID = 'app_EMoamEEZ73f0CkXaXp7hrann'
_AUTH = 'https://auth.openai.com'
SKEW_SECS = 60

`jwt_claims` decodes the access token's JWT payload to read its expiry and account details. It does not verify the signature, so these claims aren't proof of identity. `mk_jwt` makes a synthetic token for the examples; it cannot authenticate a request.

In [ ]:
def jwt_claims(token):
    "The payload of JWT `token`, decoded without verification"
    payload = token.split('.')[1]
    return json.loads(base64.urlsafe_b64decode(payload + '='*(-len(payload)%4)))

In [ ]:
def mk_jwt(**claims):
    "An unsigned JWT carrying `claims`, shaped like an access token; for examples and tests"
    enc = lambda o: base64.urlsafe_b64encode(json.dumps(o).encode()).decode().rstrip('=')
    return f"{enc(dict(alg='RS256', typ='JWT'))}.{enc(claims)}.sig"

In [ ]:
tok = mk_jwt(exp=int(time.time())+3600, **{'https://api.openai.com/auth': dict(chatgpt_plan_type='pro')})
test_eq(jwt_claims(tok)['https://api.openai.com/auth']['chatgpt_plan_type'], 'pro')
jwt_claims(tok)

{'exp': 1795515638,
 'https://api.openai.com/auth': {'chatgpt_plan_type': 'pro'}}

`token_expired` allows a 60-second margin by default (`SKEW_SECS`). It considers the token expired before its actual deadline, giving the caller time to refresh it before a request. A missing or unreadable expiry also counts as expired.

In [ ]:
def token_expired(token, skew=SKEW_SECS):
    "Is JWT `token` expired, or within `skew` seconds of expiring?"
    try: exp = jwt_claims(token)['exp']
    except (ValueError, KeyError, IndexError): return True
    return time.time() >= exp - skew

In [ ]:
test_eq(token_expired(tok), False)
test_eq(token_expired(mk_jwt(exp=int(time.time())+30)), True)
token_expired('not-a-jwt')

True

The auth file's `tokens` block holds the access, ID, and refresh tokens, plus the account ID. `codex_tokens` accepts either the full parsed file or that block alone. It requires an access token and a refresh token, then returns the block.

Treat a pasted auth file as credentials, not configuration: it gives its holder access to the signed-in account.

In [ ]:
def codex_tokens(auth):
    "The `tokens` block of an `auth.json` dict (or an already-unwrapped block), checked for the keys a refresh needs"
    if not isinstance(auth, dict): raise ValueError('Codex auth must be a JSON object')
    tokens = auth.get('tokens', auth)
    missing = [k for k in ('access_token', 'refresh_token') if not tokens.get(k)]
    if missing: raise ValueError(f"Codex auth is missing {', '.join(missing)}")
    return tokens

In [ ]:
tokens = dict(access_token=tok, refresh_token='rt_1', id_token='id_1', account_id='acct_1')
test_eq(codex_tokens(dict(auth_mode='chatgpt', tokens=tokens)), tokens)
with expect_fail(ValueError, 'missing refresh_token'): codex_tokens(dict(access_token=tok))
with expect_fail(ValueError, 'JSON object'): codex_tokens('sk-ant-oat01-x')
codex_tokens(tokens) is tokens

True

`codex_info` returns the plan and expiry claims without returning the token itself. Both fields are `None` when the token cannot be decoded. A server can use those fields to display a stored account's status, but they don't verify its current subscription entitlement.

In [ ]:
def codex_info(tokens):
    "Plan type and expiry time of the access token in `tokens`"
    try: claims = jwt_claims(tokens['access_token'])
    except (ValueError, IndexError): claims = {}
    return dict(plan=claims.get('https://api.openai.com/auth', {}).get('chatgpt_plan_type'), expires=claims.get('exp'))

In [ ]:
test_eq(codex_info(dict(access_token='not-a-jwt')), dict(plan=None, expires=None))
codex_info(tokens)

{'plan': 'pro', 'expires': 1795515638}

## Refresh

`_post` sends JSON or form data to OpenAI's auth server and raises for HTTP errors. `_token_grant` calls `/oauth/token`, then merges returned access, ID, and refresh tokens into the existing block. Fields such as `account_id` survive a refresh. A new sign-in starts with an empty block.


In [ ]:
async def _post(path, form=False, **body):
    "POST `body` to the OpenAI auth server at `path`, as a form or as JSON, returning the JSON reply"
    async with httpx2.AsyncClient() as cli: r = await cli.post(f'{_AUTH}{path}', **(dict(data=body) if form else dict(json=body)))
    r.raise_for_status()
    return r.json()

def _merge_tokens(tokens, new): return {**tokens, **{k: new[k] for k in ('access_token', 'id_token', 'refresh_token') if k in new}}

async def _token_grant(tokens, form=False, **grant):
    "The token block after `grant` at the token endpoint, merged over `tokens`; the exchange of a sign-in code is form-encoded, a refresh JSON"
    return _merge_tokens(tokens, await _post('/oauth/token', form=form, client_id=_CLIENT_ID, **grant))

`codex_refresh` sends the refresh token with Codex's public client ID and the `openid profile email` scope. Store the returned block, including any replacement refresh token.

Refresh tokens rotate. A multi-user server should serialize refreshes for each user so concurrent requests don't overwrite a newer token with an unusable one.

In [ ]:
async def codex_refresh(tokens):
    "Exchange the refresh token in `tokens` for a new token block"
    return await _token_grant(tokens, grant_type='refresh_token', refresh_token=tokens['refresh_token'], scope='openid profile email')

Call `codex_token` before sending a request. It returns `(access_token, None)` while the token is valid, without contacting the server. After a refresh it returns `(access_token, new_tokens)`. The caller must persist `new_tokens`. `force_refresh=True` skips the expiry check.

In [ ]:
async def codex_token(tokens, force_refresh=False):
    "A valid access token from `tokens`, and the refreshed block to store when a refresh happened (else `None`)"
    if not force_refresh and not token_expired(tokens['access_token']): return tokens['access_token'], None
    tokens = await codex_refresh(tokens)
    return tokens['access_token'], tokens

In [ ]:
test_eq(await codex_token(tokens), (tok, None))

Refresh requests contact OpenAI's auth server. A rejected refresh token raises `httpx2.HTTPStatusError`.

`codex_auth` reads the local auth file and calls `codex_token`. If the token refreshes, it writes the new `tokens` block back while preserving the file's other fields. Subsequent Codex CLI and `acomplete` calls can then read the replacement token.

The default path is `~/.codex/auth.json`. `CODEX_AUTH_PATH` overrides it.

In [ ]:
def codex_auth_path(): return Path(os.environ.get('CODEX_AUTH_PATH', '~/.codex/auth.json')).expanduser()

async def codex_auth(force_refresh=False):
    "A valid access token from the local Codex login, refreshing the file when needed"
    path = codex_auth_path()
    data = json.loads(path.read_text())
    token,new = await codex_token(codex_tokens(data), force_refresh)
    if new: path.write_text(json.dumps({**data, 'tokens':new}, indent=2))
    return token

Importing this module registers two Codex hooks for `acomplete`. The `vendor_auth` hook supplies the token from `codex_auth` when an auth file exists and `CODEX_AUTH_TOKEN` is unset. The `vendor_hdrs` hook sends `prompt_cache_key` as the `session-id` header, unless the caller already set one.

In [ ]:
async def _codex_vendor_auth():
    "The local Codex login's access token, or `None` without an auth file"
    return await codex_auth() if codex_auth_path().exists() else None

def _codex_hdrs(body, headers):
    "Add `prompt_cache_key` as the `session-id` header, unless `headers` already has one"
    if not (key := body.get('prompt_cache_key')) or any(k.lower() == 'session-id' for k in headers): return headers
    return {**headers, 'session-id':key}

vendor_auth['codex'],vendor_hdrs['codex'] = _codex_vendor_auth,_codex_hdrs

In [ ]:
test_eq(_codex_hdrs(dict(prompt_cache_key='k'), {}), {'session-id':'k'})
test_eq(_codex_hdrs(dict(prompt_cache_key='k'), {'Session-Id':'mine'}), {'Session-Id':'mine'})

## Sign in

Two copies of an auth file share a refresh token. Once one copy rotates it, the other can stop working. Use a separate sign-in for an independent token store rather than sharing a copied file. OpenAI's [CI authentication guide](https://developers.openai.com/codex/auth/ci-cd-auth) also explains how to persist refreshed credentials.

Device sign-in needs no browser callback on the client. `codex_device_code` requests a page URL, user code, device ID, and polling interval. The user enters the code in a browser. Run the cell below when you're ready to start a sign-in. It creates a real code that expires unused if you don't complete the flow. Notebook tests skip this request.

In [ ]:
_DEVICE = '/api/accounts/deviceauth'
async def codex_device_code():
    "Start a device sign-in: the page and code to show the user, and what the poll needs"
    r = await _post(f'{_DEVICE}/usercode', client_id=_CLIENT_ID)
    return dict(url=f'{_AUTH}/codex/device', code=r['user_code'], device_auth_id=r['device_auth_id'], interval=int(r.get('interval') or 5))

In [ ]:
#| eval: false
dev = await codex_device_code()
test_eq(set(dev), {'url', 'code', 'device_auth_id', 'interval'})
dev['url']

'https://auth.openai.com/codex/device'

`codex_device_poll` polls once at the device token endpoint: HTTP 403 and 404 return `None` while the grant isn't ready yet, other errors propagate, and once the user signs in it exchanges the authorization code for tokens using the server-provided PKCE verifier. The client doesn't generate that verifier.

`codex_device_tokens` calls `codex_device_poll` at the supplied interval with a 15-minute default timeout so a CLI wait fails instead of polling forever.

In [ ]:
async def codex_device_poll(dev):
    "Poll once for the device grant from `dev`; return the token block once approved, or `None` while pending"
    try: grant = await _post(f'{_DEVICE}/token', device_auth_id=dev['device_auth_id'], user_code=dev['code'])
    except httpx2.HTTPStatusError as e:
        if e.response.status_code in (403, 404): return None
        raise
    return await _token_grant({}, form=True, grant_type='authorization_code', code=grant['authorization_code'],
        redirect_uri=f'{_AUTH}/deviceauth/callback', code_verifier=grant['code_verifier'])

async def codex_device_tokens(dev, timeout=15*60):
    "Wait for the user to enter the code from `dev`, then exchange the grant for a token block"
    return await wait_until(lambda: codex_device_poll(dev), 'the sign-in to complete', timeout, sleep=dev['interval'])

`codex_login` combines the two device-flow steps. The `fastllm-auth` command runs it and prints the token block as JSON to stdout. Instructions go to stderr, so `fastllm-auth | pbcopy` copies only the credentials, ready for a trusted token store.

The final login example requires a person and a browser, so notebook tests skip it.

In [ ]:
async def codex_login():
    "Sign in to ChatGPT with a device code, returning the token block"
    dev = await codex_device_code()
    print(f"Open {dev['url']} and enter the code {dev['code']}", file=sys.stderr)
    return await codex_device_tokens(dev)

def main():
    "`fastllm-auth`: sign in to ChatGPT and print the token block"
    print(json.dumps(asyncio.run(codex_login())))

In [ ]:
#| eval: false
tokens = await codex_login()
codex_info(tokens)

## Completions

Pass `vendor_name='codex'`, or prefix the model name with `codex/`, to send an `acomplete` request through a ChatGPT subscription. Codex serves the Responses API from its own base URL. `acomplete` gets the token through the `vendor_auth` hook. A direct `mk_client` call needs the token passed in:

In [ ]:
cli, api_name, vendor_name = mk_client(model=luna, vendor_name='codex', api_key=await codex_auth())

In [ ]:
cli.base_url

'https://chatgpt.com/backend-api/codex'

The Codex backend accepts only streaming requests. `codex_call` streams a request with the model and system prompt these examples share, and returns the final completion:

In [ ]:
codex_sp = 'You are Codex based on GPT 5...'
async def codex_call(msgs, **kwargs):
    "Stream a Codex request and return its final completion"
    async for comp in await acomplete(msgs, model=luna, vendor_name='codex', system=codex_sp, stream=True, **kwargs): pass
    return comp

Pass a stable `prompt_cache_key` on every request in a conversation. Codex sends this key in the body and as the `session-id` header. `acomplete` does not generate a key. A `prompt_cache_key` in `xtra_body` overrides the argument. An explicit `session-id` in `xtra_hdrs` overrides the derived header. Headers apply to the request, not the cached client.

In [ ]:
comp = await codex_call([mk_msg('Hi! What are you capable of, tell concisely')], prompt_cache_key='fastllm:codex-example')
comp

I can answer questions, explain concepts, write and edit text or code, help debug, summarize documents, brainstorm ideas, and assist with research or planning. Send me a task and I’ll help.

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=32, completion_tokens=44, total_tokens=76, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 32, 'input_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'cache_write_tokens_details': {'ttl_30m_tokens': 0, 'ttl_12h_tokens': 0}}, 'output_tokens': 44, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 76, 'attribution': {'items': {'msg_0c21f3bc1d636e54016b03588cec0887d083b769b88989579d': {'input_tokens': 16, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 16, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_0c21f3bc1d636e54016b03588cec0887d0a815ac72899cc66f': {'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_0c21f3bc1d636e54016b03588d99a087d0afc1f0868abb2609': {'input_tokens': 2, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 44, 'content': [{'input_tokens': 2, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 44}]}}}})`

:::

Request visible reasoning summaries with `summary='auto'` or `summary='detailed'` in the reasoning dict. The next example passes only an effort string and does not request a summary.

In [ ]:
comp = await codex_call([mk_msg('What is 1233123 * 456231?')], reasoning_effort='medium')
comp

1233123 × 456231 = **562,588,939,413**.

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=31, completion_tokens=162, total_tokens=193, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=138, search_queries=0, raw={'input_tokens': 31, 'input_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'cache_write_tokens_details': {'ttl_30m_tokens': 0, 'ttl_12h_tokens': 0}}, 'output_tokens': 162, 'output_tokens_details': {'reasoning_tokens': 138}, 'total_tokens': 193, 'attribution': {'items': {'msg_056944731e2d04a0016b03588ef82487d09aac7267c5452890': {'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_056944731e2d04a0016b03588ef82487d0b90698509b6eecd0': {'input_tokens': 15, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 15, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_056944731e2d04a0016b035892097887d0aa2c9e1740aa3644': {'input_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 24, 'content': [{'input_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 24}]}, 'rs_056944731e2d04a0016b035891ede487d0b572920573ebf394': {'input_tokens': 2, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 138}}}})`

:::

In [ ]:
def simple_add(a:int, b:int):
    "Add two numbers"
    return a + b
sch = dict(type='function', function=get_schema(simple_add, pname='parameters'))

In [ ]:
msg1 = mk_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
comp = await codex_call([msg1], tools=[sch])
trmsg = mk_tool_res_msg(comp.tool_calls, ('8','30'))
comp = await codex_call([msg1, comp.message, trmsg], tools=[sch])
comp

3 + 5 = 8  
10 + 20 = 30

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=158, completion_tokens=19, total_tokens=177, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 158, 'input_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'cache_write_tokens_details': {'ttl_30m_tokens': 0, 'ttl_12h_tokens': 0}}, 'output_tokens': 19, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 177, 'attribution': {'items': {'fc_04b6351ee193d3b8016b035894b8f887d08113c0aa1bcf2dec': {'input_tokens': 38, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}, 'fc_04b6351ee193d3b8016b03589575cc87d0b4ec7760a5cfe3f1': {'input_tokens': 20, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}, 'fc_04b6351ee193d3b8016b035896114087d09247bbba3d847a50': {'input_tokens': 10, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 10, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'fc_04b6351ee193d3b8016b035896114087d0a9b455528bdd9088': {'input_tokens': 10, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 10, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_04b6351ee193d3b8016b035896114087d0853f7958fc25b2ef': {'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_04b6351ee193d3b8016b035896114087d09864292b694f0952': {'input_tokens': 26, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 26, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_04b6351ee193d3b8016b035896c6ec87d0ad049ce54cb4b25f': {'input_tokens': 2, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 19, 'content': [{'input_tokens': 2, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 19}]}}, 'request_fields': {'tools': {'input_tokens': 38, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}}}})`

:::

For Codex search, pass `web_search_options={'type': 'web_search'}`. The Responses adapter otherwise defaults to `web_search_preview`.

In [ ]:
comp = await codex_call([mk_msg('What is the weather in Istanbul')], web_search_options={"type":"web_search"}, tools=[sch])

In [ ]:
comp



🔧 web_search({'type': 'search', 'queries': ['weather: Istanbul, Turkey'], 'query': 'weather: Istanbul, Turkey'})
Istanbul is **partly cloudy and 52°F (11°C)** right now. Today, **Sunday, November 22**, is forecast to be sunny, with a high of **60°F (15°C)** and a low of **53°F (11°C)**. 

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=4802, completion_tokens=101, total_tokens=4903, cached_tokens=3712, cache_creation_tokens=0, reasoning_tokens=26, search_queries=0, raw={'input_tokens': 4802, 'input_tokens_details': {'cache_write_tokens': 0, 'cache_write_tokens_details': {'ttl_12h_tokens': 0, 'ttl_30m_tokens': 0}, 'cached_tokens': 3712}, 'output_tokens': 101, 'output_tokens_details': {'reasoning_tokens': 26}, 'total_tokens': 4903})`

:::

Pass a reasoning dict to request both an effort level and a visible summary:

In [ ]:
comp = await codex_call([mk_msg('What is 31231231 * 12312?')], reasoning_effort={"effort": "low", "summary": "auto"})

In [ ]:
comp


::: details
## Thinking

**Calculating the product**

:::

31231231 × 12312 = **384,518,916,072**.

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=31, completion_tokens=105, total_tokens=136, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=81, search_queries=0, raw={'input_tokens': 31, 'input_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'cache_write_tokens_details': {'ttl_30m_tokens': 0, 'ttl_12h_tokens': 0}}, 'output_tokens': 105, 'output_tokens_details': {'reasoning_tokens': 81}, 'total_tokens': 136, 'attribution': {'items': {'msg_0d80c076dc2038d8016b0358ecffd887d082fbbd3ce3f87899': {'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 14, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_0d80c076dc2038d8016b0358ecffd887d0896a43c08a3b8eba': {'input_tokens': 15, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0, 'content': [{'input_tokens': 15, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 0}]}, 'msg_0d80c076dc2038d8016b0358f1867c87d0bdb8ebccef921a2c': {'input_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 24, 'content': [{'input_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 24}]}, 'rs_0d80c076dc2038d8016b0358ee1e6087d0898f2167eddd50f7': {'input_tokens': 2, 'cached_tokens': 0, 'cache_write_tokens': 0, 'output_tokens': 81}}}})`

:::

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()